# 4.2 A Genome-Wide Association Study (GWAS) with PLINK

**Omics series, class 2 of 7.** In 4.1 we found that people with type 2 diabetes carry the **T** allele of rs7903146 a little more often than people without diabetes. Today we do what a **genome-wide association study (GWAS)** does: test **every** variant, one at a time, adjusted for age, sex, and genetic ancestry. To keep it fast and cheap, we test every variant in **one region**, around the gene *TCF7L2*, instead of the whole genome. The steps are the same.

### By the end of this notebook you'll be able to

1. Explain what a GWAS tests, and why it adjusts for covariates such as ancestry
2. Apply standard **quality-control (QC)** filters to genetic variants, and say why each one matters
3. Prepare **phenotype** and **covariate** files, and run an association test with `plink2 --glm`
4. Read GWAS results: odds ratios, p-values, and the genome-wide significance threshold
5. Make and interpret a **regional association plot** and a **Q-Q plot**, including **linkage disequilibrium (LD)**

| | |
|---|---|
| ⏱️ **Time** | 120 minutes |
| 📂 **Uses** | `gwas_samples.csv` from **4.1** (or rebuilds it), and the v9 genomic files |
| 💾 **Saves** | `gwas_tcf7l2_plink.csv`: the results, compared with Hail in **4.3** |

### Class plan

| Part | Minutes |
|---|---|
| 1. Key concepts | 25 |
| 2. Setup and samples | 10 |
| 3. Quality control | 15 |
| 4. Run the GWAS | 20 |
| 5. Read the results | 25 |
| 6. Why adjust? | 15 |
| 7. From one region to the whole genome; wrap-up | 10 |

**Before you start:**

- Run **4.1** first if you can: this notebook reuses its sample of 2,500 cases and 2,500 controls. If you haven't, it rebuilds the same sample (one cohort query, a few cents to under a dollar).
- Like 4.1, this notebook is in `notebooks/omics/` and loads `../aou_setup.R`.

> 💰 **Cost:** runs on a **standard app** (no cluster). Every PLINK step takes seconds because we read one small region. The most expensive thing is leaving the app running: stop it when you're done.

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data (including genotypes and IDs), or counts of 1 to 20. GWAS **results** for common variants (odds ratios, p-values) are summary statistics, but check the *All of Us* Data and Statistics Dissemination Policy before publishing any. Clear outputs before you share or commit this notebook (**Edit → Clear Outputs of All Cells**).

## Part 1. Key concepts

### What a GWAS does

A GWAS asks, for each variant: **is the number of copies of this allele (0, 1, or 2) linked to the trait?** It runs the same regression we ran in 4.1, once per variant:

$$\text{log odds of type 2 diabetes} = \beta_0 + \beta_1 \times \text{copies of the allele} + \beta_2 \times \text{age} + \beta_3 \times \text{sex} + \beta_4 \times \text{PC1} + \dots$$

- **Additive model:** each extra copy changes the log odds by the same amount, $\beta_1$. This is the standard GWAS model.
- **Odds ratio (OR)** $= e^{\beta_1}$: how the odds of diabetes change **per copy**. OR = 1 means no association; above 1, the allele goes with higher odds; below 1, with lower odds.
- **p-value:** how surprising the result would be if the variant had **no** effect. Small p = strong evidence of an association.

### Covariates: the other variables in the model

| Covariate | Why we adjust for it |
|---|---|
| **Age** | Type 2 diabetes becomes more common with age, and our cases are older than our controls |
| **Sex at birth** | Diabetes rates differ by sex |
| **Genetic PCs** (PC1 to PC10) | Allele frequencies **and** diabetes rates differ between ancestry groups. Without PCs, a variant can look linked to diabetes just because it's more common in a group with more diabetes (**confounding by ancestry**, 4.1 Part 4) |

### Many tests: the multiple-testing problem

If you run one test with the usual threshold of p < 0.05, there's a 5% chance of a false positive. Run 1 million tests and you'd expect **50,000** false positives. So a GWAS uses a much stricter threshold:

| Threshold | Value | Where it comes from |
|---|---|---|
| **Genome-wide significance** | p < **5 × 10⁻⁸** (0.00000005) | 0.05 divided by about 1 million independent common variants in the genome |
| **Bonferroni for our region** | p < 0.05 ÷ (number of variants we test) | the same idea, for just the variants we test today |

Bonferroni is **conservative** here: nearby variants are correlated (see LD below), so there are fewer *independent* tests than variants.

### Linkage disequilibrium (LD)

Alleles at nearby positions are often inherited together, as a block. This correlation is called **linkage disequilibrium (LD)**, measured by **r²**: 0 = unrelated, 1 = always inherited together.

So when one variant is truly linked to diabetes, its **neighbours in LD** show a signal too. A GWAS finds **regions**; working out which variant in the region actually causes the effect (**fine-mapping**) is a separate step.

### The plots

| Plot | Shows | What to look for |
|---|---|---|
| **Manhattan plot** | −log₁₀(p) for every variant, by position across the genome | "Skyscrapers": regions with very small p-values |
| **Regional plot** | the same, zoomed in on one region, coloured by LD with the top variant | a peak of variants in LD with each other |
| **Q-Q plot** | observed vs expected p-values | points on the diagonal = no signal; lifting off at the end = real associations; lifted **everywhere** = a problem, such as confounding |

On a **−log₁₀(p)** scale, small p-values become big numbers: p = 0.01 → 2; p = 5 × 10⁻⁸ → 7.3.

### Glossary

| Term | Means |
|---|---|
| **Phenotype** | the trait we study: here, case (type 2 diabetes) or control |
| **Covariate** | another variable in the model, to adjust for |
| **QC** | quality control: removing unreliable variants (or samples) before testing |
| **MAF** | minor allele frequency: the frequency of the less common allele |
| **Call rate** | the share of people with a genotype for a variant (not missing) |
| **HWE** | Hardy-Weinberg equilibrium: the genotype counts expected from the allele frequency; big departures can mean genotyping errors |
| **Summary statistics** | the results table: one row per variant, with OR and p-value |

## Part 2. Setup and samples

The same setup as 4.1. `LEAD_POS` is rs7903146's position, and the region is *TCF7L2* and the DNA around it. The gene itself runs from about 112.95 to 113.17 million (GRCh38).

In [ ]:
source("../aou_setup.R")  # settings, packages, and helper functions

AOU_DIR  <- aou_data_dir("v9")
pgen_dir <- file.path(AOU_DIR, "wgs", "short_read", "snpindel", "acaf_threshold", "pgen")

GENO_DIR <- file.path(DATA_DIR, "genomics")   # our PLINK files, in the bucket
dir.create(GENO_DIR, showWarnings = FALSE)

REGION_START <- 112700000
REGION_END   <- 113400000
GENE_START   <- 112950247   # TCF7L2, GRCh38 (from Ensembl)
GENE_END     <- 113167678
LEAD_POS     <- 112998590   # rs7903146

### The sample: 2,500 cases and 2,500 controls

If you ran **4.1**, its sample is in your bucket as `gwas_samples.csv`, and we load it. Otherwise `t2d_gwas_samples()`, a helper in `aou_setup.R`, rebuilds it with **the same steps as 4.1 Parts 3 and 4**: the cohort query, people with a genome, related people removed, ages 18 to 100, sex at birth *Female* or *Male*, a random sample, then ancestry and PCs. With the same `seed`, it picks the same people.

`col_types = cols(person_id = "c")` reads `person_id` as **text**, to match the genome IDs (4.1 Part 3).

In [ ]:
USE_SAVED_SAMPLE <- TRUE   # FALSE = build a new sample (e.g. to change its size)
N_PER_GROUP      <- 2500

if (USE_SAVED_SAMPLE && file.exists(file.path(DATA_DIR, "gwas_samples.csv"))) {
  gwas_samples <- load_data("gwas_samples.csv", col_types = cols(person_id = "c"))
  message("Loaded the sample saved in 4.1")
} else {
  gwas_samples <- t2d_gwas_samples(n_per_group = N_PER_GROUP)   # runs the cohort query
  message("Built a new sample")
}

gwas_samples %>% count(cohort)
gwas_samples %>% group_by(cohort) %>% summarise(mean_age = round(mean(age), 1),
                                                 pct_male = round(100 * mean(sex_at_birth == "Male")))

**🔍 Read the output:** 2,500 cases and 2,500 controls. Cases are **older** on average than controls, and the share of men may differ too. That's why age and sex go in the model as covariates.

### Extract the region

The same PLINK command as in 4.1: the people in our sample, and the *TCF7L2* region of chromosome 10. It takes about 10 to 30 seconds. (If you ran 4.1, this rebuilds the same files.)

In [ ]:
keep_file     <- file.path(GENO_DIR, "keep.txt")
region_prefix <- file.path(GENO_DIR, "tcf7l2")
write_lines(gwas_samples$person_id, keep_file)

plink_output <- system(str_glue(
  "plink2 --pfile {shQuote(file.path(pgen_dir, 'acaf_threshold.chr10'))} ",
  "--keep {shQuote(keep_file)} ",
  "--chr 10 --from-bp {REGION_START} --to-bp {REGION_END} ",
  "--make-pgen --out {shQuote(region_prefix)}"
), intern = TRUE)

# The key lines from PLINK's log file
str_subset(read_lines(paste0(region_prefix, ".log")), "samples|variants")

**🔍 Read the output:** `--keep: 5000 samples remaining` and about **14,000 variants** in the region, as in 4.1.

## Part 3. Quality control (QC)

Before testing, we remove variants whose results we couldn't trust. These are common GWAS filters (the same ones used in the Hail version, 4.3):

| PLINK option | Removes | Why |
|---|---|---|
| `--maf 0.01` | variants with a **minor allele frequency below 1%** in our sample | With 5,000 people, a 1% allele is carried by only about 100. Too few carriers give unstable results and no power |
| `--geno 0.1` | variants **missing in more than 10%** of people | Lots of missing genotypes often means the variant is hard to measure, and missingness can differ between cases and controls |
| `--hwe 1e-10` | variants with genotype counts **very far from HWE** | Extreme departures usually mean genotyping errors |

Notes:

- The *All of Us* data has already been through a lot of QC. The ACAF subset only has variants that are common in at least one ancestry group.
- HWE assumes one population mating at random. Our sample mixes ancestry groups, which pushes genotype counts away from HWE even when the data is fine. That's why we use a very **lenient** threshold (p < 10⁻¹⁰); many studies check HWE in controls only, within each ancestry group.
- Studies also check **samples** (e.g. people with many missing genotypes, `--mind`). *All of Us* has already removed failed samples, so we skip that here.

In [ ]:
qc_prefix <- file.path(GENO_DIR, "tcf7l2_qc")

qc_output <- system(str_glue(
  "plink2 --pfile {shQuote(region_prefix)} ",
  "--maf 0.01 --geno 0.1 --hwe 1e-10 ",
  "--make-pgen --out {shQuote(qc_prefix)}"
), intern = TRUE)

# How many variants each filter removed, and how many are left
str_subset(read_lines(paste0(qc_prefix, ".log")), "removed|remaining")

qc_pvar <- read_tsv(paste0(qc_prefix, ".pvar"), comment = "##",
                    col_types = cols(.default = "c")) %>%
  mutate(POS = as.numeric(POS))
nrow(qc_pvar)   # variants we'll test

# Is rs7903146 still there?
lead_variant <- qc_pvar %>% filter(POS == LEAD_POS)
lead_variant

**🔍 Read the output:**

- Each filter's line says how many variants it removed. Most removed variants fail `--maf`: as the histogram in 4.1 showed, most variants are **rare**.
- About half the variants are left. rs7903146 (`chr10:112998590:C`, ALT `T,G`) is still there: T is common. Its very rare G allele stays in the file, but PLINK won't be able to test it (see Part 5).

**💬 Discuss:** why not just test every variant, rare ones included? (Think about what an odds ratio means when only 3 people carry the allele.)

## Part 4. Run the GWAS

### Phenotype and covariate files

PLINK reads the phenotype and covariates from **tab-separated text files**, one row per person, matched to the genotypes by ID. The ID column must be called `#IID`, as in the `.psam` file.

| File | Columns | Notes |
|---|---|---|
| `pheno.tsv` | `#IID`, `t2d` | PLINK's code for case/control: **2 = case, 1 = control** |
| `covar.tsv` | `#IID`, `age`, `sex_male`, `PC1` … `PC10` | `sex_male`: 1 = male, 0 = female. Ten PCs is a common choice |

These files are participant-level data, so they stay in the bucket. We look only at their **column names and row counts**.

In [ ]:
pheno_file <- file.path(GENO_DIR, "pheno.tsv")
covar_file <- file.path(GENO_DIR, "covar.tsv")

pheno <- gwas_samples %>%
  transmute(`#IID` = person_id,
            t2d    = if_else(cohort == "case", 2, 1))

covar <- gwas_samples %>%
  transmute(`#IID`   = person_id,
            age,
            sex_male = if_else(sex_at_birth == "Male", 1, 0),
            across(PC1:PC10))

write_tsv(pheno, pheno_file)
write_tsv(covar, covar_file)

names(covar)
c(pheno_rows = nrow(pheno), covar_rows = nrow(covar))
sum(!complete.cases(covar))   # people missing a covariate: PLINK leaves them out

**🔍 Read the output:** 5,000 rows in each file. The last number is how many people are missing a covariate, usually because they have no PCs. PLINK leaves them out of the test (if it's 1 to 20, don't share it).

### The association test: `plink2 --glm`

| Option | Means |
|---|---|
| `--pfile …tcf7l2_qc` | the genotypes after QC |
| `--pheno pheno.tsv --pheno-name t2d` | the phenotype file, and which column to use |
| `--covar covar.tsv` | the covariates: every column except `#IID` |
| `--covar-variance-standardize` | rescales the covariates (age is in tens, PCs are tiny). Without it, PLINK can fail with a *multicollinearity* error |
| `--glm` | a **g**eneralised **l**inear **m**odel for every variant: **logistic** regression for a case/control phenotype |
| `hide-covar` | report only the variant's result, not each covariate's |
| `firth-fallback` | if the usual logistic regression fails (e.g. a rare allele), use **Firth** regression, which handles it |
| `--out …gwas_adjusted` | the start of the output file names |

It takes seconds: one regression per variant, thousands in all, on 5,000 people.

In [ ]:
gwas_prefix <- file.path(GENO_DIR, "gwas_adjusted")

glm_output <- system(str_glue(
  "plink2 --pfile {shQuote(qc_prefix)} ",
  "--pheno {shQuote(pheno_file)} --pheno-name t2d ",
  "--covar {shQuote(covar_file)} --covar-variance-standardize ",
  "--glm hide-covar firth-fallback ",
  "--out {shQuote(gwas_prefix)}"
), intern = TRUE)

str_subset(read_lines(paste0(gwas_prefix, ".log")), "samples|variants|cases|written")
list.files(GENO_DIR, pattern = "gwas_adjusted")

**🔍 Read the output:** PLINK reports how many cases and controls it used (fewer than 2,500 each if some people were missing a covariate), and writes the results to a file ending in `.glm.logistic.hybrid` (*hybrid* = logistic, with Firth where needed).

### Read the results

`read_glm()` (below) finds the results file for a run and reads it. We'll use it three times, so it's a function.

In [ ]:
# Read the results of a plink2 --glm run, given its --out prefix
read_glm <- function(prefix) {
  file <- list.files(dirname(prefix), pattern = paste0("^", basename(prefix), "\\..*glm\\."),
                     full.names = TRUE)
  read_tsv(file[1], col_types = cols(.default = "c")) %>%
    mutate(across(any_of(c("POS", "OBS_CT", "OR", "P")), as.numeric))
}

results <- read_glm(gwas_prefix)
names(results)
nrow(results)

**🔍 Read the output:** one row per variant tested (a variant with two ALT alleles has one row per allele):

| Column | Means |
|---|---|
| `#CHROM`, `POS`, `ID`, `REF`, `ALT` | the variant, as in the `.pvar` file |
| `A1` | the allele whose copies were counted: the OR is **per copy of A1** |
| `FIRTH?` | `Y` if Firth regression was used |
| `TEST` | `ADD`: the additive test of the variant |
| `OBS_CT` | the number of people in the test |
| `OR` | the odds ratio per copy of `A1`, **adjusted** for the covariates |
| `LOG(OR)_SE` | the standard error of log(OR) |
| `Z_STAT`, `P` | the test statistic and p-value |
| `ERRCODE` | `.` if the test worked; otherwise why it didn't (e.g. `CONST_ALLELE`: nobody in the sample has this allele) |

(The exact columns can differ a little between PLINK versions.)

## Part 5. Read the results

### The top variants

These are **summary statistics** (no individual's data), so they're safe to look at.

In [ ]:
results <- results %>% filter(!is.na(P))   # drop tests that didn't work (ERRCODE)

N_TESTS          <- nrow(results)
REGION_THRESHOLD <- 0.05 / N_TESTS   # Bonferroni for this region
GENOME_THRESHOLD <- 5e-8             # genome-wide significance

c(tests = N_TESTS, region_threshold = signif(REGION_THRESHOLD, 2))

results %>%
  arrange(P) %>%
  select(ID, POS, REF, ALT, A1, OR, P) %>%
  mutate(OR = round(OR, 2), P = signif(P, 2)) %>%
  head(10)

**🔍 Read the output:**

- The **smallest p-values** are at the top. Are they near rs7903146 (position 112,998,590), inside *TCF7L2* (112.95 to 113.17 million)?
- Do any pass the **region** threshold? The **genome-wide** threshold (5 × 10⁻⁸)?
- The top variants usually have similar ORs and positions: they're in **LD** with each other (next).

### rs7903146 itself

In [ ]:
results %>%
  filter(POS == LEAD_POS) %>%
  select(ID, REF, ALT, A1, OBS_CT, OR, P) %>%
  mutate(rank = map_int(P, ~ sum(results$P <= .x)))   # 1 = the smallest p-value in the region

**🔍 Read the output:** the row with `A1` = **T** is the risk allele: an **adjusted** OR per copy of T, and its **rank** among all the variants tested. In 4.1, the unadjusted OR was about **1.17**. Is the adjusted one similar? (Part 6 looks at this.)

There may also be a row for `A1` = G that didn't get a p-value: G is so rare that almost no one in our sample has it.

**💬 Discuss:** with 5,000 people, rs7903146 probably **doesn't** reach genome-wide significance, even though it's the best-known type 2 diabetes variant. Why not? (Hint: the published finding came from tens of thousands of people. A small effect, such as OR ≈ 1.2, needs a large sample to give a tiny p-value. This is **statistical power**.)

### Regional association plot

A regional plot shows −log₁₀(p) along the region. We colour each variant by its **LD (r²) with rs7903146**, so we can see whether the signal is one LD block.

To get r², we export everyone's genotypes in the region with `--export A` (as in 4.1, but every variant) and calculate the correlation of each variant with rs7903146. The file goes to the app's **temporary folder** (`tempdir()`), not the bucket: we only need it for a minute, and it's deleted when the app stops.

| Function | What it does |
|---|---|
| `cor(x, y)^2` | r², the squared correlation between two variants' genotypes |
| `str_remove(…, "_[^_]+$")` | turns a column name like `chr10:112998590:C_C` back into the variant ID |
| `cut()` | groups r² into bands for the colours |

In [ ]:
ld_prefix <- file.path(tempdir(), "tcf7l2_dosages")

export_output <- system(str_glue(
  "plink2 --pfile {shQuote(qc_prefix)} --export A --out {shQuote(ld_prefix)}"
), intern = TRUE)

dosages <- read_tsv(paste0(ld_prefix, ".raw"),
                    col_types = cols(.default = "d", FID = "c", IID = "c"))
geno    <- as.matrix(select(dosages, -(FID:PHENOTYPE)))   # people x variants
dim(geno)

lead_id  <- lead_variant$ID[1]
variants <- str_remove(colnames(geno), "_[^_]+$")
lead_col <- which(variants == lead_id)[1]

ld <- tibble(ID = variants,
             r2 = as.numeric(cor(geno[, lead_col], geno, use = "pairwise.complete.obs")^2)) %>%
  distinct(ID, .keep_all = TRUE)

rm(dosages, geno)   # free the memory: these are big

plot_data <- results %>%
  left_join(ld, by = "ID") %>%
  mutate(ld_band = cut(r2, breaks = c(0, 0.2, 0.4, 0.6, 0.8, 1), include.lowest = TRUE,
                       labels = c("0-0.2", "0.2-0.4", "0.4-0.6", "0.6-0.8", "0.8-1")))

ggplot(plot_data, aes(POS / 1e6, -log10(P), colour = ld_band)) +
  geom_point(size = 1.5) +
  geom_point(data = filter(plot_data, POS == LEAD_POS), shape = 23, size = 4,
             fill = "#d7301f", colour = "black") +
  geom_hline(yintercept = -log10(GENOME_THRESHOLD), linetype = "dashed") +
  geom_hline(yintercept = -log10(REGION_THRESHOLD), linetype = "dotted") +
  annotate("segment", x = GENE_START / 1e6, xend = GENE_END / 1e6, y = -0.4, yend = -0.4,
           linewidth = 2, colour = "grey30") +
  annotate("text", x = (GENE_START + GENE_END) / 2e6, y = -0.9, label = "TCF7L2",
           fontface = "italic", size = 3.5) +
  scale_colour_manual(values = c("#9ecae1", "#6baed6", "#3182bd", "#08519c", "#08306b"),
                      na.value = "grey70", drop = FALSE) +
  theme_bw() +
  labs(x = "Position on chromosome 10 (millions, GRCh38)", y = "-log10(p)",
       colour = "LD (r²) with\nrs7903146",
       title = "Type 2 diabetes association in the TCF7L2 region",
       subtitle = "Diamond: rs7903146. Dashed line: genome-wide threshold. Dotted line: region threshold.")

**🔍 Read the output:**

- **The peak:** are the variants with the smallest p-values (highest points) **inside the *TCF7L2* gene** (the bar at the bottom)?
- **The colours:** variants in strong LD with rs7903146 (dark blue) should rise and fall **together** with it. That's one association signal, seen through many correlated variants.
- **The lines:** points above the dashed line would be genome-wide significant; above the dotted line, significant after correcting for this region only.
- If the highest point **isn't** rs7903146, that's normal in a small sample: variants in strong LD carry almost the same information, and chance decides which comes out on top. Fine-mapping methods, with much larger samples, sort this out.

LD **differs between ancestry groups**: the blocks are usually shorter in people of African ancestry. Our r² mixes all groups, so it's an average.

👩‍🏫 **Instructor tip:** this style of plot is often called a **LocusZoom** plot, after the tool that popularised it. Ask students to point out the peak, the gene, and the lead variant before explaining the colours.

### ✍️ Your turn

How many variants pass the **region** threshold? How many pass the **genome-wide** threshold?

<details>
<summary><b>Show answer</b></summary>

```r
results %>%
  summarise(pass_region = sum(P < REGION_THRESHOLD),
            pass_genome = sum(P < GENOME_THRESHOLD))
```

With 5,000 people you'll likely see a few (or none) pass the region threshold, and none pass the genome-wide one. **Not significant doesn't mean no effect:** we know from much larger studies that this region matters. It means **this sample is too small** to show it at such a strict threshold.

</details>

In [ ]:
# Your code here

## Part 6. Why adjust? Adjusted vs unadjusted

Let's run the same test **without** covariates and compare. `allow-no-covars` tells PLINK that leaving out covariates is deliberate.

In [ ]:
unadj_prefix <- file.path(GENO_DIR, "gwas_unadjusted")

unadj_output <- system(str_glue(
  "plink2 --pfile {shQuote(qc_prefix)} ",
  "--pheno {shQuote(pheno_file)} --pheno-name t2d ",
  "--glm allow-no-covars firth-fallback ",
  "--out {shQuote(unadj_prefix)}"
), intern = TRUE)

results_unadj <- read_glm(unadj_prefix) %>% filter(!is.na(P))

# rs7903146 (T allele): unadjusted vs adjusted
bind_rows(unadjusted = results_unadj, adjusted = results, .id = "model") %>%
  filter(POS == LEAD_POS, A1 == "T") %>%
  select(model, OBS_CT, OR, P)

**🔍 Read the output:** the **unadjusted** OR should match 4.1's (about 1.17; the number of people can differ slightly). Adjusting for age, sex, and ancestry changes it. Ancestry matters most here: the T allele's frequency differs between ancestry groups (the exercise at the end of 4.1), and so does the share of cases.

### The Q-Q plot and genomic inflation (λ)

A **Q-Q plot** compares the p-values we got with the p-values we'd **expect if no variant had any effect** (evenly spread between 0 and 1).

**λ (lambda, the genomic inflation factor)** summarises it: the median test statistic divided by the median expected under no effect.

| λ | Means |
|---|---|
| about **1** | p-values look as expected: no sign of confounding |
| clearly **above 1** (e.g. 1.2) | p-values are too small **across the board**: often confounding by ancestry, or relatedness |

⚠️ λ is meant for the **whole genome**, where almost all variants have no effect. We picked a region **because** it has a real signal, and its variants are in LD, so λ here will be off. We use it to **compare** the two models, not to judge either one.

In [ ]:
qq_data <- bind_rows(unadjusted = results_unadj, adjusted = results, .id = "model") %>%
  filter(TEST == "ADD") %>%
  group_by(model) %>%
  arrange(P, .by_group = TRUE) %>%
  mutate(expected = -log10(ppoints(n())),
         observed = -log10(P)) %>%
  ungroup()

qq_data %>%
  group_by(model) %>%
  summarise(lambda = round(median(qchisq(P, df = 1, lower.tail = FALSE)) / qchisq(0.5, df = 1), 2))

ggplot(qq_data, aes(expected, observed, colour = model)) +
  geom_abline(slope = 1, intercept = 0, colour = "grey50") +
  geom_point(size = 1.2, alpha = 0.7) +
  scale_colour_manual(values = c(adjusted = "#2171b5", unadjusted = "#d95f02")) +
  theme_bw() +
  labs(x = "Expected -log10(p), if no variant had an effect", y = "Observed -log10(p)",
       colour = "Model", title = "Q-Q plot: TCF7L2 region")

**🔍 Read the output:**

- Points **on the grey line** behave as expected under no effect. Points **above** it have smaller p-values than chance would give.
- Compare the two models: is the **unadjusted** one lifted higher, and its λ larger? That extra lift is partly **confounding by ancestry**: variants whose frequency differs between ancestry groups look associated with diabetes because diabetes rates differ too.
- Both curves rise at the right end: that's the real *TCF7L2* signal (and its LD).

👩‍🏫 **Instructor tip:** in a genome-wide analysis, λ above about 1.05 to 1.1 is a warning sign. For very large samples with many true small effects, λ is naturally above 1; methods such as **LD score regression** separate real signal from confounding.

### ✍️ Your turn

Run the adjusted GWAS on people of **European genetic ancestry** (`ancestry == "eur"`) only. Is rs7903146's OR closer to the published 1.3 to 1.4? How has the p-value changed, and why?

<details>
<summary><b>Show answer</b></summary>

```r
eur_keep   <- file.path(GENO_DIR, "keep_eur.txt")
eur_prefix <- file.path(GENO_DIR, "gwas_eur")
write_lines(gwas_samples %>% filter(ancestry == "eur") %>% pull(person_id), eur_keep)

eur_output <- system(str_glue(
  "plink2 --pfile {shQuote(qc_prefix)} --keep {shQuote(eur_keep)} ",
  "--pheno {shQuote(pheno_file)} --pheno-name t2d ",
  "--covar {shQuote(covar_file)} --covar-variance-standardize ",
  "--glm hide-covar firth-fallback --out {shQuote(eur_prefix)}"
), intern = TRUE)

read_glm(eur_prefix) %>%
  filter(POS == LEAD_POS, A1 == "T") %>%
  select(OBS_CT, OR, P)
```

The OR is often closer to the published value, since most published studies were in people of European ancestry. But the **p-value is usually larger**: fewer people (about 3,000), so less power. Analysing each ancestry group separately and then combining the results (**meta-analysis**) is a common approach in diverse cohorts like *All of Us*.

</details>

In [ ]:
# Your code here

## Save the results

The results table has one row per variant, and no individual's data. We save it to the bucket, to compare with the **Hail** results in 4.3.

In [ ]:
save_data(results, "gwas_tcf7l2_plink.csv")

## Part 7. From one region to the whole genome

Everything we did works genome-wide: drop `--chr 10 --from-bp … --to-bp …`, and run the same steps **for each chromosome**. But the scale changes:

| | Today | Genome-wide, short-read genomes |
|---|---|---|
| Variants tested | thousands | tens of millions |
| Data read | a few MB | the ACAF files for all 22 chromosomes: terabytes (chromosome 10 alone is 80 GB) |
| Time | seconds | many hours, on a much bigger app or a cluster |
| Cost | cents | can be tens to hundreds of dollars: estimate on a small piece first |

Options for a real genome-wide study in *All of Us*:

- **The genotyping arrays** (`v9/microarray`): about 1.9 million variants instead of tens of millions, so much smaller files. A common first GWAS.
- **REGENIE** or **SAIGE**: programs built for large biobank GWAS. They handle related people and case/control imbalance, so you can keep everyone instead of sampling.
- **Hail** (Python, **4.3**): reads the Hail-format genomes and can spread the work over a cluster.
- **Results that already exist:** the *All of Us* **All by All** browser has GWAS results for thousands of phenotypes. We use it in **4.4**.

👩‍🏫 **Instructor tip:** before running anything genome-wide, time and cost one chromosome (chromosome 22 is the smallest) and multiply. Remind students that the genomic bucket is *Requester Pays*.

## ⚠️ Common errors

PLINK's exact wording varies between versions.

| Message | Means |
|---|---|
| *Error: No phenotypes present* / *--pheno-name: 't2d' not found* | The phenotype file's header isn't `#IID t2d`, or its IDs don't match the genotype IDs |
| *--glm: … multicollinearity* or *VIF* error | Covariates are too correlated, or on very different scales. Check `--covar-variance-standardize` is in the command |
| *No covariates loaded; … allow-no-covars* | PLINK refuses to run without covariates unless you add `allow-no-covars` |
| `read_glm()` error: *'file' must be a character string* | No results file: PLINK failed. Look at `glm_output` (or the `.log` file) for its error message |
| Results for rs7903146 are empty | Check `LEAD_POS`, and that the variant passed QC (`lead_variant` in Part 3) |
| *cannot allocate vector* in the LD step | The app ran out of memory. Use a bigger app, or export a smaller window around rs7903146 with `--from-bp`/`--to-bp` |

## ✅ Key takeaways

- A GWAS runs **one regression per variant**: copies of the allele vs the trait, **adjusted** for covariates
- **QC first:** remove rare, poorly measured, and suspicious variants
- **Adjust for genetic ancestry** with PCs; compare models with a **Q-Q plot** and **λ**
- With millions of tests, the threshold is **p < 5 × 10⁻⁸**; small samples have little **power** for small effects
- Nearby variants move together (**LD**): a GWAS finds **regions**, not necessarily the causal variant

**Next:** **4.3 gwas_hail**: the same analysis in Python with Hail, the tool *All of Us* uses for its own genome-wide analyses.

**Before you share or commit this notebook:** clear all outputs (**Edit → Clear Outputs of All Cells**).